# Workshop 2 — Silver Data Exploration

## 1. Introduction and Objectives

This notebook examines saved Silver Parquet outputs for NOAA HazEL historical tsunami observations. Bronze preserves the source responses; `src/silver.py` validates, types, deduplicates and records lineage in Silver. `src/load.py` maps Silver records to the canonical PostgreSQL model, but this notebook does not query the database. The goal is to inspect schema, missingness, quality, outliers, history and consistency across processing runs. Only one snapshot supplies the main descriptive results.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sample_dir = project_root / "datalake_silver" / "samples"
if not sample_dir.is_dir():
    raise FileNotFoundError(f"Silver sample directory not found: {sample_dir}. Run from the repository root or notebooks/.")

## 2. Data Discovery and Loading

Discover the saved tsunami Parquet outputs through the repository-relative `datalake_silver/samples/` path. The earliest named snapshot is the representative dataset; later files are examined separately in Section 8. No snapshots are concatenated.

In [ ]:
snapshot_paths = sorted(sample_dir.glob("tsunami_events_*.parquet"))
if not snapshot_paths:
    raise FileNotFoundError(f"No tsunami_events_*.parquet files found in {sample_dir}")
snapshot_path = snapshot_paths[0]
tsunamis = pd.read_parquet(snapshot_path)
print("Available snapshots:", *[p.name for p in snapshot_paths], sep="\n  ")
print(f"Representative snapshot: {snapshot_path.name} ({len(tsunamis)} rows)")

## 3. Dataset Overview and Schema Inspection

Inspect the record count, date coverage, source, types and Parquet schema. In `src/silver.py`, NOAA dates remain separate from full timestamps when clock components are incomplete. `event_time` is therefore nullable; `event_date`, `time_precision`, source identifiers, lineage fields and `content_hash` are expected Silver fields.

In [ ]:
print(f"Shape: {tsunamis.shape[0]} rows × {tsunamis.shape[1]} columns")
print("Source counts:")
display(tsunamis["source"].value_counts().rename_axis("source").to_frame("rows"))
print(f"Event-date coverage: {tsunamis['event_date'].min()} to {tsunamis['event_date'].max()}")
print("Time precision:")
display(tsunamis["time_precision"].value_counts(dropna=False).rename_axis("precision").to_frame("rows"))

In [ ]:
schema = pd.DataFrame({"column": tsunamis.columns, "pandas_dtype": tsunamis.dtypes.astype(str).values})
display(schema)
print(f"Parquet rows: {pq.ParquetFile(snapshot_path).metadata.num_rows}")
print("Parquet schema:")
print(pq.read_schema(snapshot_path))
expected_fields = {"source", "external_id", "event_date", "event_time", "time_precision",
                   "latitude", "longitude", "reported_eq_magnitude", "max_water_height",
                   "max_water_height_iqr_outlier", "bronze_file", "bronze_manifest", "content_hash"}
missing_fields = sorted(expected_fields - set(tsunamis.columns))
if missing_fields:
    raise ValueError(f"Expected Silver fields are missing: {missing_fields}")

## 4. Missing-Value Analysis

Nulls are retained rather than imputed. NOAA records can have only a date or partial clock time; the Silver processor leaves `event_time` null unless seconds are known. Missing earthquake magnitude and depth can also reflect a non-earthquake tsunami cause, as observed in Workshop 1. These are different from invalid values.

In [ ]:
missing = pd.DataFrame({"missing": tsunamis.isna().sum()})
missing["percent"] = (missing["missing"] / len(tsunamis) * 100).round(1)
display(missing.sort_values(["missing", "percent"], ascending=False))

In [ ]:
cause_missing = pd.crosstab(tsunamis["cause_code"].eq(1).map({True: "earthquake cause (code 1)", False: "other cause"}),
                            tsunamis["reported_eq_magnitude"].isna(), rownames=["cause group"],
                            colnames=["reported earthquake magnitude missing"])
display(cause_missing)
print("Incomplete timestamps:", int(tsunamis["event_time"].isna().sum()))
print("Missing coordinates:", int((tsunamis["latitude"].isna() | tsunamis["longitude"].isna()).sum()))

## 5. Data Quality and Validation

Check current-snapshot identifiers, valid geographic ranges, date/precision consistency, numeric plausibility and source lineage. A missing coordinate or measurement is reported separately from an invalid non-null value. These checks inspect Silver output; they do not re-run the normalization pipeline.

In [ ]:
duplicate_keys = int(tsunamis.duplicated(["source", "external_id"]).sum())
invalid_lat = int((tsunamis["latitude"].notna() & ~tsunamis["latitude"].between(-90, 90)).sum())
invalid_lon = int((tsunamis["longitude"].notna() & ~tsunamis["longitude"].between(-180, 180)).sum())
invalid_numeric = {field: int(tsunamis[field].lt(0).sum()) for field in
                   ["reported_eq_magnitude", "reported_eq_depth", "max_water_height", "num_runups"]}
checks = pd.Series({"duplicate (source, external_id) keys": duplicate_keys,
                    "invalid non-null latitudes": invalid_lat,
                    "invalid non-null longitudes": invalid_lon,
                    **{f"negative {key}": value for key, value in invalid_numeric.items()}})
display(checks.to_frame("count"))

In [ ]:
dated = pd.to_datetime(tsunamis["event_date"], errors="coerce")
timed = tsunamis["event_time"].notna()
time_checks = pd.Series({
    "unparseable or missing event dates": int(dated.isna().sum()),
    "full timestamps with different event date": int((timed & (tsunamis["event_time"].dt.date != dated.dt.date)).sum()),
    "full timestamps with incomplete precision": int((timed & ~tsunamis["time_precision"].isin(["second", "subsecond"])).sum()),
    "partial precision with a full timestamp": int((~timed & tsunamis["time_precision"].isin(["second", "subsecond"])).sum()),
})
display(time_checks.to_frame("count"))

In [ ]:
lineage_fields = ["bronze_file", "locator", "bronze_manifest", "content_hash"]
lineage = pd.DataFrame({"missing": tsunamis[lineage_fields].isna().sum(),
                        "blank": tsunamis[lineage_fields].astype(str).apply(lambda col: col.str.strip().eq("").sum())})
display(lineage)
print("Invalid SHA-256 content hashes:", int((~tsunamis["content_hash"].astype(str).str.fullmatch(r"[0-9a-f]{64}")).sum()))
print("Bronze files:", tsunamis["bronze_file"].nunique(),
      "| Bronze manifests:", tsunamis["bronze_manifest"].nunique())

In [ ]:
rejected_paths = sorted(sample_dir.glob("rejected_records_*.parquet"))
if rejected_paths:
    print("Saved rejected-record outputs:")
    for path in rejected_paths:
        rejected = pd.read_parquet(path)
        print(f"{path.name}: {len(rejected)} rows")
        if not rejected.empty:
            display(rejected.head())
else:
    print("No rejected-record Parquet sample is available; quarantine contents cannot be assessed here.")

## 6. Outlier Analysis

`src/silver.py` flags non-null heights outside $[Q_1 - 1.5\,IQR,\;Q_3 + 1.5\,IQR]$. It preserves flagged observations. The flag is a statistical screening result, not evidence that an extreme tsunami measurement is wrong.

In [ ]:
height = tsunamis["max_water_height"].dropna()
q1, q3 = height.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
recomputed = tsunamis["max_water_height"].notna() & ((tsunamis["max_water_height"] < lower) | (tsunamis["max_water_height"] > upper))
print(f"Non-null heights: {len(height)}; Q1={q1:g} m, Q3={q3:g} m, IQR={iqr:g} m")
print(f"IQR bounds: {lower:g} to {upper:g} m")
print("Stored flags:", int(tsunamis["max_water_height_iqr_outlier"].sum()),
      "| Flag disagreements:", int((recomputed != tsunamis["max_water_height_iqr_outlier"]).sum()))
display(tsunamis.loc[tsunamis["max_water_height_iqr_outlier"],
                     ["external_id", "event_date", "max_water_height", "country", "location_name"]]
        .sort_values("max_water_height", ascending=False).head(10))

## 7. Historical Data Exploration

The following plots use the representative 221-record snapshot only. Counts reflect saved NOAA observations, not a complete census of all Pacific tsunamis. Height is in meters; `reported_eq_magnitude` is the source's associated earthquake magnitude and is absent for some events. No magnitude scale is inferred for records without a documented type.

In [ ]:
yearly = pd.to_datetime(tsunamis["event_date"]).dt.year.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 3.5))
yearly.plot.bar(ax=ax, color="steelblue")
ax.set(title="Saved NOAA tsunami observations by year", xlabel="Event year", ylabel="Observations")
fig.tight_layout()
plt.show()
print(f"Yearly counts range from {yearly.min()} to {yearly.max()} across {len(yearly)} years.")

In [ ]:
shown_height = height[height <= 2]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(shown_height, bins=20, color="teal", edgecolor="white")
ax.set(title="Maximum reported water height (0–2 m detail)", xlabel="Maximum water height (m)", ylabel="Observations")
fig.tight_layout()
plt.show()
print(f"Measured heights: {len(height)}; median {height.median():g} m; maximum {height.max():g} m.")
print(f"Plot shows {len(shown_height)} values up to 2 m; {len(height) - len(shown_height)} larger values remain in the analysis.")

In [ ]:
magnitudes = tsunamis["reported_eq_magnitude"].dropna()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(magnitudes, bins=20, color="slateblue", edgecolor="white")
ax.set(title="Reported associated earthquake magnitude", xlabel="Reported earthquake magnitude (source value)", ylabel="Observations")
fig.tight_layout()
plt.show()
print(f"Reported magnitudes: {len(magnitudes)}; range {magnitudes.min():g}–{magnitudes.max():g}. Missing values are excluded from this plot only.")

## 8. Silver Processing Runs Comparison

Compare each file independently by row count, column schema, `(source, external_id)` and `content_hash`. Lineage and processing timestamps may differ between runs even when the scientific observations and their content hashes agree. This checks Silver-output consistency, **not** PostgreSQL idempotency or successful database loading.

In [ ]:
snapshots = {path.name: pd.read_parquet(path) for path in snapshot_paths}
summary = pd.DataFrame([
    {"snapshot": name, "rows": len(frame), "columns": len(frame.columns),
     "unique_keys": frame[["source", "external_id"]].drop_duplicates().shape[0],
     "unique_hashes": frame["content_hash"].nunique(),
     "bronze_manifests": frame["bronze_manifest"].nunique()}
    for name, frame in snapshots.items()
]).set_index("snapshot")
display(summary)

In [ ]:
reference = tsunamis.set_index(["source", "external_id"])["content_hash"]
comparison = []
for name, frame in snapshots.items():
    indexed = frame.set_index(["source", "external_id"])["content_hash"]
    shared = reference.index.intersection(indexed.index)
    comparison.append({"snapshot": name,
                       "same_column_schema": frame.columns.tolist() == tsunamis.columns.tolist() and frame.dtypes.astype(str).tolist() == tsunamis.dtypes.astype(str).tolist(),
                       "same_keys": reference.index.equals(indexed.index) or set(reference.index) == set(indexed.index),
                       "shared_keys": len(shared),
                       "different_hashes_on_shared_keys": int((reference.loc[shared] != indexed.loc[shared]).sum())})
display(pd.DataFrame(comparison).set_index("snapshot"))

## 9. Canonical Database Model Readiness

The mappings below follow `src/load.py` and `sql/01_schema.sql`; they describe intended loading, not observed PostgreSQL rows.

| Silver field(s) | Canonical destination | Meaning |
|---|---|---|
| `source`, `external_id` | `event.source`, `event.external_id` | Current observation key, unique per source. |
| `event_time`, `event_date`, `known_hour`, `known_minute`, `time_precision` | `event.timestamp`, date and precision fields | Preserves partial NOAA times; complete times use the Silver UTC interpretation, whose source timezone still needs confirmation. |
| `latitude`, `longitude` | `event.latitude`, `event.longitude` | Nullable WGS84 coordinates. |
| `max_water_height` | `event.severity` and `tsunami_event.max_water_height` | Maximum reported height, in meters; `event.severity_scale` is `tsunami_max_water_height`. |
| `cause_code`, `event_validity`, `region_code`, `warning_status_id` | `tsunami_event` and NOAA reference catalogs | Original NOAA coded descriptors. |
| `reported_eq_magnitude`, `reported_eq_depth`, `num_runups`, `country`, `location_name` | `tsunami_event` | Source attributes; depth is in km, water height is in m. |
| `bronze_file`, `locator`, `content_hash` | `event.raw_reference`, `event.content_hash` | File and record lineage plus change detection. |

`bronze_manifest` and `silver_processed_at` describe the Silver run; the current loader does not insert them as `event` columns. The IQR flag is a Silver quality indicator and is not a `tsunami_event` column.

## 10. Conclusions

The saved samples contain three separately readable 221-row, 28-column Silver outputs. The representative file covers 2010-01-02 through 2025-12-12. Its `(source, external_id)` keys are unique, all non-null coordinates fall within valid ranges, and the saved lineage fields and hashes are populated. Thirty-two records lack a complete timestamp, nine lack coordinates, and 12 lack maximum water height; these nulls remain unfilled. Of 209 measured heights, 33 carry the existing IQR flag, including extreme values that are retained for scientific review.

The three saved files have the same keys, schemas and per-key content hashes; they represent repeated processing of the same 221 observations, not 663 unique events. The Bronze filenames and manifests differ between runs. This notebook cannot establish whether PostgreSQL accepted those records or whether database upserts were idempotent. No rejected-record Parquet sample is available here, so quarantine contents are also outside this inspection.